# S01 — Verified Agent Loops
Northstar Support Systems (synthetic). Open this notebook from the repo root's shared
Jupyter environment (`uv sync --group s01 && uv run jupyter lab` from the repo root) — the
kernel's working directory automatically matches this notebook's own folder, so
`triage/` and `data/` resolve with no extra setup. The default mode is offline and
deterministic.


## 1. 💻 First gratification: classify a ticket
A mock adapter simulates a model decision. It is **not a real LLM**. Compare direct classification with tool-assisted classification.

In [ ]:
import json
from pathlib import Path
from triage.core import MockModel, run_agent, verify
tickets = json.loads(Path("data/tickets.json").read_text())
tickets

In [ ]:
direct = run_agent(tickets[0], MockModel())
print(json.dumps(direct, indent=2))

> [!TIP]
> **💡 Think:** Is this an agent? Does a single model decision require tools? Why not?

## 2. 🔍 Observe the agent loop
The model may *request* `lookup_account`; only application code can authorize and execute it.

In [ ]:
tool_run = run_agent(tickets[1], MockModel())
print(json.dumps(tool_run, indent=2))

> [!TIP]
> **🔍 Locate in `triage/core.py`:** tool allowlist, account scoping, maximum steps, repeated-call detection, and the verifier.

## 3. 🧩 Verification and feedback
The next mock run deliberately proposes an invalid category first. The verifier rejects it and feeds errors back to the next decision.

In [ ]:
recovery = run_agent(tickets[0], MockModel(inject_invalid_first=True))
print(json.dumps(recovery, indent=2))

In [ ]:
escalation = run_agent(tickets[0], MockModel(inject_invalid_first=True), max_steps=1)
print(json.dumps(escalation, indent=2))

> [!TIP]
> **💭 Reflect:** Why is valid JSON not enough? What can this verifier prove, and what can it not prove?

## 4. 🧩 Loop Engineering
- **Agent loop:** decide / act / observe.
- **Verification:** independent checks.
- **Event-driven continuation:** tool results or verifier failures cause the next step.
- **Hill-climbing:** compare measurable changes against a fixed test set; do not equate retries with optimization.

> [!NOTE]
> This is a **course-defined teaching framework**.

## 5. 🛠️ Extension tasks
1. Implement a test that rejects unauthorized tools.
2. Add a test for repeated tool requests.
3. Create a small labeled dataset and report accuracy and escalation rate.
4. Explain when a deterministic rule or single model call would be better than this loop.

In [ ]:
assert direct["status"] == "accepted"
assert tool_run["status"] == "accepted"
assert any(event["event"] == "tool_executed" for event in tool_run["trace"])
assert any(event["event"] == "verification_failed" for event in recovery["trace"])
assert escalation["status"] == "needs_review"
print("Notebook checks passed")